# Wilcoxon Signed-Rank Tests

Paired tests across subjects, as described in Methods 3.10. Each subject gives
one balanced accuracy per setting, so the two settings are compared subject by
subject.

This notebook only reads CSV files that `file1.ipynb` and
`external_validation.ipynb` already saved. It does not retrain any model.

**Rules used**
- Two-sided test, alpha = 0.05.
- Subjects with exactly zero difference are dropped, as in the standard
  Wilcoxon procedure. The column `n_used` shows how many subjects remain.
- Holm correction is applied within each family of tests. Both raw and
  corrected p values are reported.

## 0. Paths and helper functions

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# Folder with the CSV files from file1.ipynb
WESAD_OUTPUT_DIR = Path(".")
# Folder with the CSV files from external_validation.ipynb
EXTERNAL_OUTPUT_DIR = Path("outputs_external")

STATS_OUTPUT_DIR = Path("outputs_stats")
STATS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = ["DT", "LDA", "RF"]
SIGNAL_ORDER = ["BVP", "EDA", "TEMP", "ACC"]
TOL = 1e-10

In [2]:
def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test of a minus b.

    a and b are balanced accuracies of the same subjects, in the same order.
    Returns a dictionary with the summary and the test result.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    diff_pp = 100 * (a - b)

    nonzero = np.abs(diff_pp) > TOL * 100
    result = {
        "n_subjects": len(diff_pp),
        "n_used": int(nonzero.sum()),
        "mean_diff_pp": diff_pp.mean(),
        "median_diff_pp": np.median(diff_pp),
        "n_higher": int((diff_pp > TOL * 100).sum()),
        "n_equal": int((~nonzero).sum()),
        "n_lower": int((diff_pp < -TOL * 100).sum()),
        "W": np.nan,
        "p_value": np.nan,
    }

    if result["n_used"] == 0:
        return result  # all differences are zero; the test is undefined

    test = wilcoxon(diff_pp[nonzero], alternative="two-sided")
    result["W"] = float(test.statistic)
    result["p_value"] = float(test.pvalue)
    return result


def holm(p_values):
    """Holm-adjusted p values. NaN values are left as NaN."""
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full_like(p, np.nan)
    valid = np.flatnonzero(~np.isnan(p))
    if len(valid) == 0:
        return adjusted

    order = valid[np.argsort(p[valid])]
    m = len(order)
    running_max = 0.0
    for rank, idx in enumerate(order):
        value = min(1.0, (m - rank) * p[idx])
        running_max = max(running_max, value)
        adjusted[idx] = running_max
    return adjusted


def finish_table(rows, name):
    """Add Holm correction, a significance flag, round and save."""
    table = pd.DataFrame(rows)
    table["p_holm"] = holm(table["p_value"])
    table["significant_holm"] = table["p_holm"] < 0.05
    table.to_csv(STATS_OUTPUT_DIR / f"{name}.csv", index=False)
    return table.round({"mean_diff_pp": 2, "median_diff_pp": 2, "W": 1,
                        "p_value": 4, "p_holm": 4})


def paired_columns(df, subject_col, a_col, b_col):
    """Return two aligned arrays, checking that no subject is missing."""
    df = df.dropna(subset=[a_col, b_col]).sort_values(subject_col)
    return df[a_col].to_numpy(), df[b_col].to_numpy()

## 1. WESAD: calibrated versus uncalibrated

Positive differences favor calibration. Corresponds to Results Table 4.

In [3]:
wesad_calibration = pd.read_csv(WESAD_OUTPUT_DIR / "exploratory_calibration_paired_changes.csv")

rows = []
for model in MODEL_ORDER:
    sub = wesad_calibration.loc[wesad_calibration["model"] == model]
    a, b = paired_columns(sub, "held_out_subject", "Calibrated_2min", "Uncalibrated")
    rows.append({"model": model, **paired_wilcoxon(a, b)})

wesad_calibration_test = finish_table(rows, "wilcoxon_wesad_calibration")
display(wesad_calibration_test)

,model,n_subjects,n_used,mean_diff_pp,median_diff_pp,n_higher,n_equal,n_lower,W,p_value,p_holm,significant_holm
0,DT,15,11,11.50,11.76,9,4,2,6.5,0.0146,0.0439,True
1,LDA,15,13,-0.36,0.00,6,2,7,42.5,0.8528,0.8528,False
2,RF,15,12,4.64,4.55,10,3,2,16.0,0.0723,0.1445,False


After Holm correction, two-minute personal calibration significantly improved the DT model compared with the uncalibrated model (mean difference: +11.50 percentage points; median difference: +11.76 percentage points; Holm-adjusted *p* = 0.0439). Performance improved for 9 of the 15 subjects, remained unchanged for 4, and decreased for 2. The calibration effects for LDA and RF were not statistically significant after correction.

## 2. WESAD: full model versus each signal removed

Positive differences mean the full model is better, so the removed signal
added information. Corresponds to Results Table 5. One family of 12 tests.

In [4]:
wesad_ablation = pd.read_csv(WESAD_OUTPUT_DIR / "exploratory_ablation_paired_changes.csv")

rows = []
for signal in SIGNAL_ORDER:
    for model in MODEL_ORDER:
        sub = wesad_ablation.loc[
            (wesad_ablation["removed_signal"] == signal)
            & (wesad_ablation["model"] == model)
        ]
        a, b = paired_columns(sub, "held_out_subject",
                              "full_balanced_accuracy", "balanced_accuracy")
        rows.append({"removed_signal": signal, "model": model, **paired_wilcoxon(a, b)})

wesad_ablation_test = finish_table(rows, "wilcoxon_wesad_ablation")
display(wesad_ablation_test)

,removed_signal,model,n_subjects,n_used,mean_diff_pp,median_diff_pp,n_higher,n_equal,n_lower,W,p_value,p_holm,significant_holm
0,BVP,DT,15,4,2.12,0.00,4,11,0,0.0,0.1250,1.0000,False
1,BVP,LDA,15,12,7.08,0.00,7,3,5,23.5,0.2393,1.0000,False
2,BVP,RF,15,3,-0.30,0.00,2,12,1,3.0,1.0000,1.0000,False
3,EDA,DT,15,10,10.57,10.88,10,5,0,0.0,0.0020,0.0234,True
4,EDA,LDA,15,12,4.65,1.34,8,3,4,23.5,0.2412,1.0000,False
5,EDA,RF,15,11,7.55,5.00,9,4,2,17.5,0.1807,1.0000,False
6,TEMP,DT,15,9,9.45,2.94,8,6,1,4.0,0.0273,0.2734,False
7,TEMP,LDA,15,12,3.12,0.00,6,3,6,28.5,0.4351,1.0000,False
8,TEMP,RF,15,6,7.08,0.00,5,9,1,2.0,0.0938,0.8438,False
9,ACC,DT,15,1,1.82,0.00,1,14,0,0.0,1.0000,1.0000,False


Removing EDA significantly reduced the performance of the DT model after Holm correction: the full model exceeded the model without EDA by a mean of 10.57 percentage points and a median of 10.88 percentage points (Holm-adjusted *p* = 0.0234). All 10 subjects with a non-zero difference favored the full model, while the remaining 5 showed no change. None of the other signal-removal comparisons remained statistically significant after correction.

## 3. WESAD: five-minute versus two-minute calibration

Positive differences favor five minutes. Corresponds to Results Table 6.

In [5]:
wesad_duration = pd.read_csv(
    WESAD_OUTPUT_DIR / "sensitivity_calibration_duration_paired_changes.csv"
)

rows = []
for model in MODEL_ORDER:
    sub = wesad_duration.loc[wesad_duration["model"] == model]
    a, b = paired_columns(sub, "held_out_subject",
                          "Calibration_5min_common", "Calibration_2min_common")
    rows.append({"model": model, **paired_wilcoxon(a, b)})

wesad_duration_test = finish_table(rows, "wilcoxon_wesad_calibration_length")
display(wesad_duration_test)

,model,n_subjects,n_used,mean_diff_pp,median_diff_pp,n_higher,n_equal,n_lower,W,p_value,p_holm,significant_holm
0,DT,15,5,0.29,0.0,3,10,2,7.0,1.0000,1.0000,False
1,LDA,15,7,1.12,0.0,4,8,3,9.0,0.4688,0.9375,False
2,RF,15,3,2.26,0.0,3,12,0,0.0,0.2500,0.7500,False


## 4. WESAD: feature-quality sensitivity

Two families, matching Results Table 7:
- 28 features versus 19 features, within each calibration setting.
- Calibrated versus uncalibrated, within the 19-feature set.

In [6]:
quality_pairs = pd.read_csv(WESAD_OUTPUT_DIR / "quality_sensitivity_calibration_pairs.csv")

feature_sets = list(quality_pairs["feature_set"].unique())
print("Feature sets found:", feature_sets)

full_name, reduced_name = "Full_28", "Reduced_19"
assert {full_name, reduced_name} <= set(feature_sets)
print(f"Full set: {full_name} | Reduced set: {reduced_name}")

Feature sets found: ['Full_28', 'Reduced_19']
Full set: Full_28 | Reduced set: Reduced_19


In [7]:
rows = []
for configuration in ["Uncalibrated", "Calibrated_2min"]:
    wide = quality_pairs.pivot_table(
        index=["model", "held_out_subject"], columns="feature_set", values=configuration
    ).reset_index()
    for model in MODEL_ORDER:
        sub = wide.loc[wide["model"] == model]
        a, b = paired_columns(sub, "held_out_subject", full_name, reduced_name)
        rows.append({"configuration": configuration, "model": model,
                     "comparison": "28 minus 19 features", **paired_wilcoxon(a, b)})

quality_feature_test = finish_table(rows, "wilcoxon_wesad_feature_quality_28_vs_19")
display(quality_feature_test)

rows = []
for model in MODEL_ORDER:
    sub = quality_pairs.loc[
        (quality_pairs["feature_set"] == reduced_name) & (quality_pairs["model"] == model)
    ]
    a, b = paired_columns(sub, "held_out_subject", "Calibrated_2min", "Uncalibrated")
    rows.append({"model": model, "comparison": "calibrated minus uncalibrated, 19 features",
                 **paired_wilcoxon(a, b)})

quality_calibration_test = finish_table(rows, "wilcoxon_wesad_feature_quality_calibration")
display(quality_calibration_test)

,configuration,model,comparison,n_subjects,n_used,mean_diff_pp,median_diff_pp,n_higher,n_equal,n_lower,W,p_value,p_holm,significant_holm
0,Uncalibrated,DT,28 minus 19 features,15,12,2.54,4.71,9,3,3,23.0,0.2334,0.5000,False
1,Uncalibrated,LDA,28 minus 19 features,15,13,11.52,12.94,12,2,1,7.0,0.0046,0.0278,True
2,Uncalibrated,RF,28 minus 19 features,15,9,4.46,0.00,7,6,2,9.0,0.1289,0.5000,False
3,Calibrated_2min,DT,28 minus 19 features,15,4,2.12,0.00,4,11,0,0.0,0.1250,0.5000,False
4,Calibrated_2min,LDA,28 minus 19 features,15,11,8.96,0.00,7,4,4,13.0,0.0830,0.4150,False
5,Calibrated_2min,RF,28 minus 19 features,15,4,0.56,0.00,3,11,1,3.0,0.6250,0.6250,False


,model,comparison,n_subjects,n_used,mean_diff_pp,median_diff_pp,n_higher,n_equal,n_lower,W,p_value,p_holm,significant_holm
0,DT,"calibrated minus uncalibrated, 19 features",15,14,11.92,13.64,11,1,3,18.0,0.0295,0.0886,False
1,LDA,"calibrated minus uncalibrated, 19 features",15,13,2.20,4.55,9,2,4,34.0,0.4548,0.4548,False
2,RF,"calibrated minus uncalibrated, 19 features",15,11,8.55,7.06,9,4,2,10.0,0.0420,0.0886,False


In the uncalibrated setting, LDA performed significantly better with the full 28-feature set than with the reduced 19-feature set (mean difference: +11.52 percentage points; median difference: +12.94 percentage points; Holm-adjusted *p* = 0.0278). The 28-feature model performed better for 12 subjects, was unchanged for 2, and performed worse for 1. No other 28-versus-19-feature comparison was significant after correction, and calibration within the 19-feature set was also not significant for any model after correction.

## 5. External: calibrated versus uncalibrated

33 PhysioNet subjects. Positive differences favor calibration.

In [8]:
external_calibration = pd.read_csv(EXTERNAL_OUTPUT_DIR / "external_calibration_pairs.csv")

rows = []
for model in MODEL_ORDER:
    sub = external_calibration.loc[external_calibration["model"] == model]
    a, b = paired_columns(sub, "subject_id", "Calibrated_2min", "Uncalibrated")
    rows.append({"model": model, **paired_wilcoxon(a, b)})

external_calibration_test = finish_table(rows, "wilcoxon_external_calibration")
display(external_calibration_test)

,model,n_subjects,n_used,mean_diff_pp,median_diff_pp,n_higher,n_equal,n_lower,W,p_value,p_holm,significant_holm
0,DT,33,31,-5.87,-5.56,12,2,19,165.5,0.1059,0.2119,False
1,LDA,33,29,-1.72,0.00,16,4,13,198.0,0.6732,0.6732,False
2,RF,33,31,-11.24,-10.42,8,2,23,118.0,0.0097,0.0292,True


In the external PhysioNet cohort, two-minute personal calibration significantly reduced RF performance (mean difference: -11.24 percentage points; median difference: -10.42 percentage points; Holm-adjusted *p* = 0.0292). Calibration improved performance for 8 subjects, produced no change for 2, and reduced performance for 23. The calibration effects for DT and LDA were not statistically significant after correction.

## 6. External: full model versus each signal removed

Calibrated features, 33 subjects. Positive differences mean the full model is
better. Negative differences mean removing the signal helped.

In [9]:
external_ablation = pd.read_csv(EXTERNAL_OUTPUT_DIR / "external_ablation_subject_scores.csv")

wide = external_ablation.pivot_table(
    index=["model", "subject_id"], columns="feature_set", values="balanced_accuracy"
).reset_index()

rows = []
for signal in SIGNAL_ORDER:
    for model in MODEL_ORDER:
        sub = wide.loc[wide["model"] == model]
        a, b = paired_columns(sub, "subject_id", "Full", f"Without_{signal}")
        rows.append({"removed_signal": signal, "model": model, **paired_wilcoxon(a, b)})

external_ablation_test = finish_table(rows, "wilcoxon_external_ablation")
display(external_ablation_test)

,removed_signal,model,n_subjects,n_used,mean_diff_pp,median_diff_pp,n_higher,n_equal,n_lower,W,p_value,p_holm,significant_holm
0,BVP,DT,33,12,4.08,0.00,11,21,1,6.0,0.0068,0.0752,False
1,BVP,LDA,33,24,4.03,1.79,17,9,7,99.0,0.1450,1.0000,False
2,BVP,RF,33,8,-0.69,0.00,3,25,5,10.0,0.3125,1.0000,False
3,EDA,DT,33,32,2.02,4.55,21,1,11,217.5,0.3846,1.0000,False
4,EDA,LDA,33,23,6.88,1.92,17,10,6,56.0,0.0126,0.1261,False
5,EDA,RF,33,30,1.82,3.03,17,3,13,208.0,0.6143,1.0000,False
6,TEMP,DT,33,23,-6.27,0.00,7,10,16,61.0,0.0179,0.1609,False
7,TEMP,LDA,33,22,1.18,0.00,13,11,9,114.0,0.6847,1.0000,False
8,TEMP,RF,33,16,-5.86,0.00,5,17,11,16.0,0.0052,0.0619,False
9,ACC,DT,33,1,0.38,0.00,1,32,0,0.0,1.0000,1.0000,False
